In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# 1. 单帧模型 (Single Frame)
# ============================================================
class SingleFrameModel(nn.Module):
    """
    核心思想: 每帧独立处理 -> 特征平均 -> 分类
    特点: 完全忽略时序，只看静态图像
    """
    def __init__(self, num_classes=10):
        super().__init__()
        # 2D CNN，只能处理单张图像
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(64, num_classes)
    
    def forward(self, video):
        # T是帧数,3D结构数据
        B, T, C, H, W = video.shape
        # 每一帧独立通过CNN
        frame_scores = []
        for t in range(T):
            frame = video[:, t, :, :, :]  
            feat = self.cnn(frame)         
            feat = feat.view(B, -1)       
            score = self.fc(feat)         
            frame_scores.append(score)
        # 所有帧的结果取平均
        outputs = torch.stack(frame_scores, dim=1)  
        final = outputs.mean(dim=1)                 
        return final


# ============================================================
# 2. 晚期融合 (Late Fusion)
# ============================================================
class LateFusionModel(nn.Module):
    """
    核心思想: 两帧独立提取特征 -> 特征拼接 -> 分类
    特点: 只在最后的全连接层合并信息
    """
    def __init__(self, num_classes=10, fusion_frames=2):
        super().__init__()
        self.fusion_frames = fusion_frames
        
        # 2D CNN，每帧独立提取特征
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        
        # 拼接后的特征维度: 64 * fusion_frames
        self.fc = nn.Linear(64 * fusion_frames, num_classes)
    
    def forward(self, video):
        B, T, C, H, W = video.shape
        # 选择几帧（取均匀分布）
        indices = torch.linspace(0, T-1, self.fusion_frames).long()
        
        features = []
        for idx in indices:
            frame = video[:, idx, :, :, :]  
            feat = self.cnn(frame)          
            feat = feat.view(B, -1)          
            features.append(feat)
        # 特征拼接
        combined = torch.cat(features, dim=1)
        self.fc(combined)#分类         
        
        return output


# ============================================================
# 3. 早期融合 (Early Fusion)
# ============================================================
class EarlyFusionModel(nn.Module):
    """
    核心思想: 在输入层就将多帧叠加 -> 3D卷积直接处理时空
    特点: 把时间维度当成额外的通道或维度
    """
    def __init__(self, num_classes=10, fusion_frames=8):
        super().__init__()
        self.fusion_frames = fusion_frames
        # 方法A: 通道叠加 (将多帧在通道维度堆叠)
        self.cnn_channel_stack = nn.Sequential(
            nn.Conv2d(3 * fusion_frames, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        # 方法B: 3D卷积，保持时间维度
        self.conv3d = nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(3, 3, 3), padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        
        self.fc = nn.Linear(64, num_classes)
    
    def forward_channel_stack(self, video):
        B, T, C, H, W = video.shape  
        
        #在通道维度上拼接所有帧
        video_stacked = video.view(B, C * T, H, W)  
        #view函数用于重塑张量形状
        feat = self.cnn_channel_stack(video_stacked)  
        feat = feat.view(B, -1)                       
        output = self.fc(feat)
        return output
    
    def forward_conv3d(self, video):
        B, T, C, H, W = video.shape
        video_3d = video.permute(0, 2, 1, 3, 4)  
        #permute函数重排维度顺序，使卷积核在时间上滑动
        feat = self.conv3d(video_3d)  
        feat = feat.view(B, -1)       
        output = self.fc(feat)
        return output
    
    def forward(self, video):
        # 选择一种方式
        return self.forward_conv3d(video)


# ============================================================
# 4. 慢速融合 / SlowFast (Slow Fusion)
# ============================================================
    """
    核心思想: 双路并行，不同时间分辨率 -> 多阶段横向连接
    特点: Slow路看空间细节(低帧率)，Fast路看运动变化(高帧率)
    """
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Slow路径: 低帧率，强空间建模
        self.slow_path = nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3)),
            nn.ReLU(),
            nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        
        # Fast路径: 高帧率，强时间建模
        self.fast_path = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3)),
            nn.ReLU(),
            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        
        # 横向连接: 将Fast特征投影到Slow通道空间
        self.lateral_conv = nn.Conv3d(64, 128, kernel_size=1)
        
        # 分类器
        self.fc = nn.Linear(128 + 64, num_classes)
    
    def forward(self, video):
        # video: (B, T, C, H, W)
        B, T, C, H, W = video.shape
        
        # Slow路径: 取慢帧率 (比如每秒2帧)
        slow_indices = torch.arange(0, T, 4)  # 16帧取4帧
        slow_video = video[:, slow_indices, :, :, :]  # (B, T_slow, C, H, W)
        slow_video = slow_video.permute(0, 2, 1, 3, 4)  # (B, C, T_slow, H, W)
        slow_feat = self.slow_path(slow_video)  # (B, 128, 1, 1, 1)
        slow_feat = slow_feat.view(B, -1)       # (B, 128)
        
        # Fast路径: 取高帧率 (保留更多帧)
        fast_video = video.permute(0, 2, 1, 3, 4)  # (B, C, T, H, W)
        fast_feat = self.fast_path(fast_video)     # (B, 64, 1, 1, 1)
        fast_feat = fast_feat.view(B, -1)          # (B, 64)
        
        # 横向连接: Fast信息注入Slow
        # （实际需要更多的横向连接，这里是简化版）
        
        # 融合两个路径的特征
        combined = torch.cat([slow_feat, fast_feat], dim=1)  # (B, 192)
        output = self.fc(combined)  # (B, num_classes)
        
        return output


# ============================================================
# 使用示例
# ============================================================
if __name__ == "__main__":
    # 模拟输入: 批次4，16帧，3通道，112x112
    video = torch.randn(4, 16, 3, 112, 112)
    
    # 创建模型
    single = SingleFrameModel(num_classes=10)
    late = LateFusionModel(num_classes=10, fusion_frames=2)
    early = EarlyFusionModel(num_classes=10, fusion_frames=8)
    slowfast = SlowFastModel(num_classes=10)
    
    # 前向传播
    out1 = single(video)   # (4, 10)
    out2 = late(video)     # (4, 10)
    out3 = early(video)    # (4, 10)
    out4 = slowfast(video) # (4, 10)
    
    print("单帧模型输出:", out1.shape)
    print("晚期融合输出:", out2.shape)
    print("早期融合输出:", out3.shape)
    print("慢速融合输出:", out4.shape)

In [ ]:
import torch
import torch.nn as nn
from mypath import Path

class C3D(nn.Module):
    """
    The C3D network.
    """

    def __init__(self, num_classes, pretrained=False):
        super(C3D, self).__init__()

        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

        self.conv2 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool4 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool5 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2), padding=(0, 1, 1))

        self.fc6 = nn.Linear(8192, 4096)
        self.fc7 = nn.Linear(4096, 4096)
        self.fc8 = nn.Linear(4096, num_classes)

        self.dropout = nn.Dropout(p=0.5)

        self.relu = nn.ReLU()

        self.__init_weight()

        if pretrained:
            self.__load_pretrained_weights()

    def forward(self, x):

        x = self.relu(self.conv1(x))
        x = self.pool1(x)

        x = self.relu(self.conv2(x))
        x = self.pool2(x)

        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x))
        x = self.pool3(x)

        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x))
        x = self.pool4(x)

        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x))
        x = self.pool5(x)

        x = x.view(-1, 8192)
        x = self.relu(self.fc6(x))
        x = self.dropout(x)
        x = self.relu(self.fc7(x))
        x = self.dropout(x)

        logits = self.fc8(x)

        return logits

    def __load_pretrained_weights(self):
        """Initialiaze network."""
        corresp_name = {
                        # Conv1
                        "features.0.weight": "conv1.weight",
                        "features.0.bias": "conv1.bias",
                        # Conv2
                        "features.3.weight": "conv2.weight",
                        "features.3.bias": "conv2.bias",
                        # Conv3a
                        "features.6.weight": "conv3a.weight",
                        "features.6.bias": "conv3a.bias",
                        # Conv3b
                        "features.8.weight": "conv3b.weight",
                        "features.8.bias": "conv3b.bias",
                        # Conv4a
                        "features.11.weight": "conv4a.weight",
                        "features.11.bias": "conv4a.bias",
                        # Conv4b
                        "features.13.weight": "conv4b.weight",
                        "features.13.bias": "conv4b.bias",
                        # Conv5a
                        "features.16.weight": "conv5a.weight",
                        "features.16.bias": "conv5a.bias",
                         # Conv5b
                        "features.18.weight": "conv5b.weight",
                        "features.18.bias": "conv5b.bias",
                        # fc6
                        "classifier.0.weight": "fc6.weight",
                        "classifier.0.bias": "fc6.bias",
                        # fc7
                        "classifier.3.weight": "fc7.weight",
                        "classifier.3.bias": "fc7.bias",
                        }

        p_dict = torch.load(Path.model_dir())
        s_dict = self.state_dict()
        for name in p_dict:
            if name not in corresp_name:
                continue
            s_dict[corresp_name[name]] = p_dict[name]
        self.load_state_dict(s_dict)

    def __init_weight(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                # n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                # m.weight.data.normal_(0, math.sqrt(2. / n))
                torch.nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm3d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

def get_1x_lr_params(model):
    """
    This generator returns all the parameters for conv and two fc layers of the net.
    """
    b = [model.conv1, model.conv2, model.conv3a, model.conv3b, model.conv4a, model.conv4b,
         model.conv5a, model.conv5b, model.fc6, model.fc7]
    for i in range(len(b)):
        for k in b[i].parameters():
            if k.requires_grad:
                yield k

def get_10x_lr_params(model):
    """
    This generator returns all the parameters for the last fc layer of the net.
    """
    b = [model.fc8]
    for j in range(len(b)):
        for k in b[j].parameters():
            if k.requires_grad:
                yield k

if __name__ == "__main__":
    inputs = torch.rand(1, 3, 16, 112, 112)
    net = C3D(num_classes=101, pretrained=True)

    outputs = net.forward(inputs)
    print(outputs.size())